In [1]:
import numpy as np
import pandas as pd

In [ ]:
visits = pd.read_csv(r'C:visits_and_wait_time_c.csv')

In [21]:
visits['visit_date'] = pd.to_datetime(visits['visit_date'], format='mixed')
visits['check_in_time'] = pd.to_datetime(visits['check_in_time'], format='mixed')
visits['seen_by_provider_time'] = pd.to_datetime(visits['seen_by_provider_time'], format='mixed')

In [22]:
print(visits.shape)
print(visits.dtypes)
visits.head()

(20000, 9)
visit_id                            str
patient_id                          str
visit_date               datetime64[us]
visit_type                          str
department                          str
check_in_time            datetime64[us]
seen_by_provider_time    datetime64[us]
hcahps_score                    float64
provider_id                         str
dtype: object


,visit_id,patient_id,visit_date,visit_type,department,check_in_time,seen_by_provider_time,hcahps_score,provider_id
0,VIS-000001,PAT-01825,2017-05-21,ER,Pediatrics,2017-05-21 09:47:00,2017-05-21 10:41:00,76.5,PRV-0140
1,VIS-000002,PAT-01425,2021-01-17,ER,Oncology,2021-01-17 13:01:00,2021-01-17 14:42:00,81.8,PRV-0144
2,VIS-000003,PAT-03258,2022-12-26,Outpatient,Oncology,2022-12-26 14:51:00,2022-12-26 15:13:00,69.2,PRV-0002
3,VIS-000004,PAT-02616,2022-06-24,Outpatient,Pediatrics,2022-06-24 16:06:00,2022-06-24 16:52:00,81.2,PRV-0024
4,VIS-000005,PAT-06225,2018-06-26,Outpatient,Pulmonology,2018-06-26 16:34:00,2018-06-26 16:54:00,84.0,PRV-0032


In [27]:
# Calculate wait time in minutes
visits['wait_time'] = (
    visits['seen_by_provider_time'] - visits['check_in_time']).dt.total_seconds() / 60

In [30]:
visits['wait_time'].describe()

count    19000.000000
mean        36.830947
std         34.254829
min          5.000000
25%         17.000000
50%         25.000000
75%         43.000000
max        360.000000
Name: wait_time, dtype: float64

In [52]:
# Flag long waits
perc_90 = visits['wait_time'].quantile(0.90)
visits['long_wait'] = visits['wait_time'] > perc_90

# percentage of longwaits
print((visits['long_wait'].mean() * 100).round(2))

9.4


In [53]:
visits.groupby('long_wait').size()

long_wait
False    18121
True      1879
dtype: int64

In [59]:
# HCAHPS score distribution

visits['hcahps_bins'] = pd.cut(
        visits['hcahps_score'],
        bins = [0, 40, 60, 80, 100],
        labels = ['Poor (0-40)', 'Fair (40-60)', 'Good (60-80)', 'Excellent (80-100)'])

visits['hcahps_bins'].value_counts()

hcahps_bins
Good (60-80)          9916
Excellent (80-100)    5238
Fair (40-60)          3602
Poor (0-40)            444
Name: count, dtype: int64

In [73]:
# YoY wait time and satisfaction analysis
visits['year'] = visits['visit_date'].dt.year

visits.groupby('year').agg(
    avg_wait=('wait_time', 'mean'),
    avg_hcahps=('hcahps_score', 'mean'),
    visit_count=('visit_id', 'count')
).round(2)

,avg_wait,avg_hcahps,visit_count
year,,,
2017,28.71,77.89,1807
2018,28.73,77.39,2074
2019,28.22,77.37,2155
2020,56.23,59.64,4018
2021,44.18,65.26,3551
2022,28.25,72.30,2860
2023,29.05,77.51,1955
2024,27.95,77.11,1580


In [74]:
# Correlation between wait time and HCAHPS score
visits[['wait_time', 'hcahps_score']].corr()

,wait_time,hcahps_score
wait_time,1.000000,-0.279297
hcahps_score,-0.279297,1.000000


In [75]:
visits.groupby('long_wait')['hcahps_score'].agg(
    ['mean', 'median', 'count']
).round(2)

,mean,median,count
long_wait,,,
False,72.19,73.0,17413
True,59.11,58.8,1788


In [91]:
# Review score distribution by Departments
visits.groupby('department').agg(
    total_visits = ('hcahps_score', 'count'),
    avg_score = ('hcahps_score', 'mean'),
    bad_score = ('hcahps_score', lambda x : (x <= 50).sum(), )
).round(2).sort_values('bad_score', ascending=False)

,total_visits,avg_score,bad_score
department,,,
Pulmonology,2588,67.58,320
General,3180,70.37,282
Neurology,1853,71.37,144
Cardiology,2125,71.93,143
Psychiatry,1644,70.71,124
Pediatrics,1604,71.84,123
Orthopedics,1805,71.74,121
Oncology,1615,71.92,103
Dermatology,1455,71.77,99
